In [ ]:
''' 
* first 19 pages doesnt need remove them 

1) 74 - > end extract all using PyMuPDF4LLMLoader 
2) pages 73 , 236 extract using the pymupdf

first pages with sentencee with the newline char or the paragraph chars
clean them list them 


so split document more for this 
then after this then use the markdown splitter
 
then create those chunks differently



3) extract tables for ipc -> bns

'''

In [3]:
# 8 pages separate pdf crop then process
# inclusive 74 to  82 inclusive pages 


import pymupdf

src = pymupdf.open(r"D:\LegalSaathi AI\data\pdfs\BNS2023.pdf")

main_file = pymupdf.open()  # new empty pdf

main_file.insert_pdf(src, from_page=74, to_page=81)

main_file.save("docs/from_page_74_to_81.pdf")

main_file.close()


In [1]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

loader = PyMuPDF4LLMLoader(
    file_path="D:/LegalSaathi AI/src/notebooks/docs/from_page_74_to_81.pdf",
    mode="single",  
)
docs = loader.load()

d:\LegalSaathi AI\legalAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
clean_text = ''

import re

page_split =docs[0].page_content.split('\n\n\n')

temp_page = []
for line in page_split:
    line = re.sub(r'\n|—|–|-----', ' ', line)
    line = re.sub(r'\*\*|––|;', '', line)
    temp_page.append(line)
clean_text +=' '.join(temp_page)


In [3]:
pattern = r'(\(\d+\))'
pieces = re.split(pattern, clean_text)
small_chunks = []
for i in range(1, len(pieces), 2):
    small_chunks.append(pieces[i] + pieces[i+1])
    
    
small_chunks = [txt for txt in small_chunks if len(txt) >= 20]

In [4]:
small_chunks

['(1) “act” denotes as well a series of acts as a single act ',
 '(2) “animal” means any living creature, other than a human being ',
 '(3) “child” means any person below the age of eighteen years ',
 '(4) “counterfeit”. A person is said to “counterfeit” who causes one thing to resemble another thing, intending by means of that resemblance to practise deception, or knowing it to be likely that deception will thereby be practised. _Explanation_ 1. It is not essential to counterfeiting that the imitation should be exact. _Explanation_ 2. When a person causes one thing to resemble another thing, and the resemblance is such that a person might be deceived thereby, it shall be presumed, until the contrary is proved, that the person so causing the one thing to resemble the other thing intended by means of that resemblance to practise deception or knew it to be likely that deception would thereby be practised ',
 '(5) “Court” means a Judge who is empowered by law to act judicially alone, or a

# main pages

In [ ]:
# 8 pages separate pdf crop then process
# inclusive 74 to  82 inclusive pages 


import pymupdf

src = pymupdf.open(r"D:\LegalSaathi AI\data\pdfs\BNS2023.pdf")

main_file = pymupdf.open()  # new empty pdf

main_file.insert_pdf(src, from_page=82, to_page=237)

main_file.save("docs/page_82_to_237.pdf")

main_file.close()


In [12]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

Loader = PyMuPDF4LLMLoader(
    file_path=r'D:\LegalSaathi AI\src\notebooks\docs\page_82_to_237.pdf',
    mode='page'
)

docs = Loader.load()

In [13]:
for i, doc in enumerate(docs):
    char_count = len(doc.page_content.strip())
    if char_count < 20:
        print(f"Page index {i} (PDF page {doc.metadata.get('page', i)}): {char_count} characters")

Page index 154 (PDF page 154): 0 characters


In [14]:
import pymupdf
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
pdf_path= r'D:\LegalSaathi AI\src\notebooks\docs\page_82_to_237.pdf'
loader = PyMuPDF4LLMLoader(pdf_path)
docs = loader.load()

src = pymupdf.open(pdf_path)

for i, doc in enumerate(docs):
    char_count = len(doc.page_content.strip())
    if char_count < 20:
        print(f"Page index {i}: {char_count} characters -> using pymupdf fallback")
        doc.page_content = src[i].get_text().strip()

src.close()

# Merge all pages into one single string, in original order
full_text = "\n".join(doc.page_content.strip() for doc in docs)


Page index 154: 0 characters -> using pymupdf fallback


In [15]:
import re
# convert the section bold to [section] for easy title breakdown dont collide
converted_text = re.sub(r'\*\*(\d+)\.\*\*', r'[\1]', full_text)

In [16]:
# solves this problem
# OF FALSE EVIDENCE AND OFFENCES AGAINST PUBLIC
# JUSTICE

converted_text = converted_text.replace('**\n\n**',' ')
converted_text = converted_text.replace('**\n**',' ')
converted_text = converted_text.replace('** **',' ')

In [17]:
import re

text = re.sub(r'\*\*([^*]+)\*\*', r'## \1', converted_text)

with open("docs/full_text.md", "w", encoding="utf-8") as f:
    f.write(text)

In [18]:
import re

# load once
with open("docs/full_text.md", "r", encoding="utf-8") as f:
    text = f.read()

# Step 1: convert ## 14. or ## 14 . → [14]
text = re.sub(r'##\s*(\d+)\s*\.', r'[\1]', text)

# Step 2: fix ## stuck to previous text — inject newline
text = re.sub(r'([^\n])##', r'\1\n\n##', text)

# Step 3: ALL CAPS ## lines → # (chapter level)
text = re.sub(r'^##\s+([A-Z][A-Z\s,\.]+)$', r'# \1', text, flags=re.MULTILINE)

# Step 4: bare ALL CAPS lines with no ## → add #
text = re.sub(r'^([A-Z][A-Z\s,\.]{3,})$', r'# \1', text, flags=re.MULTILINE)


# save
with open("docs/full_text.md", "w", encoding="utf-8") as f:
    f.write(text)

print("Done")

Done


In [ ]:
import re

# >>> set your file path here <
file_path = r"D:\LegalSaathi AI\src\notebooks\docs\full_text.md"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# NEW_____________________________________

# make CHAPTER X, XI, XII etc → # (H1), case-insensitive, any number of leading #'s


text = re.sub(r'^#{1,6}\s*(CHAPTER\s+[IVXLCDM\d]+.*)$', r'# \1', text, flags=re.MULTILINE | re.IGNORECASE)

# make OF CONTEMPTS... style subheadings → ## (H2) — but only if they're ALL CAPS
# and NOT already a section title
text = re.sub(r'^#\s+([A-Z][A-Z\s,\.]+)$', r'## \1', text, flags=re.MULTILINE)

# demote section titles from ## to ###
# section titles are mixed case (not ALL CAPS)
text = re.sub(
    r'^##\s+([A-Z][a-z][^\n]+)$',   # starts with Capital then lowercase = mixed case
    r'### \1',
    text,
    flags=re.MULTILINE
)

text = re.sub(
    r'^#{1,6}\s*(\d+)\s*\.\s*(.+)$',
    r'[\1] \2',
    text,
    flags=re.MULTILINE
)


# converrt  ## CHAPTER -> # CHAPTER

text = text.replace("\r\n", "\n").replace("\r", "\n")
pattern = re.compile(
    r'^#{1,6}[ \t]*(CHAPTER\s+[IVXLCDM\d]+.*)$',
    flags=re.MULTILINE | re.IGNORECASE
)
matches = pattern.findall(text)

text, n = pattern.subn(r'# \1', text)


with open(file_path, "w", encoding="utf-8") as f:
    f.write(text)

print(f"Done. Updated headings written to {file_path}")

Done. Updated headings written to D:\LegalSaathi AI\src\notebooks\docs\full_text.md


# chunking

In [1]:
with open(file=r'D:\LegalSaathi AI\data\output\bns\main_clean.md',encoding='utf8') as file:
    md_text = file.read()

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(
     headers_to_split_on=[
        ("#",   "chapter"),          # CHAPTER XIII
        ("##",  "chapter_title"),    # OF CONTEMPTS OF THE LAWFUL...
        ("###", "section_title"),    # Absconding to avoid...
    ],
    strip_headers=True
)

docs = splitter.split_text(md_text)

d:\LegalSaathi AI\legalAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(docs[11].page_content)

[14] Nothing is an offence which is done by a person who is, or who
by reason of a mistake of fact and not by reason of a mistake of law in good
faith believes himself to be, bound by law to do it.  
_Illustrations_ .  
(a) A, a soldier, fires on a mob by the order of his superior officer, in
conformity with the commands of the law. A has committed no offence.  
(b) A, an officer of a Court, being ordered by that Court to arrest Y,
and, after due enquiry, believing Z to be Y, arrests Z. A has committed no
offence.


In [4]:
print(docs[11].metadata)

{'chapter': 'CHAPTER III', 'chapter_title': 'GENERAL EXCEPTIONS', 'section_title': 'Act done by a person bound, or by mistake of fact believing himself bound, by law.'}


In [6]:
docs[11].metadata['section_no'] = 32

In [7]:
print(docs[11].metadata)

{'chapter': 'CHAPTER III', 'chapter_title': 'GENERAL EXCEPTIONS', 'section_title': 'Act done by a person bound, or by mistake of fact believing himself bound, by law.', 'section_no': 32}


In [16]:
from sentence_transformers import SentenceTransformer
import chromadb

Embedding_model = SentenceTransformer(model_name_or_path='sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3869.36it/s]


In [17]:
Embedding_model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [18]:
client = chromadb.Client()
db = client.get_or_create_collection(name='legal_passages')

In [19]:
db

Collection(name=legal_passages)

## PArent-Child Chunking /  add the data to vector db (db) and parent datain (parent_data)

### sections data

In [24]:
'''

CODE TO CREATE THE JSON FORMAT FILE FOR PARENT CHILD CHUNKS

'''
import re
parent_data = {}

def clean_text(text):
    text = text.replace('\n\n', ' ')
    text = text.replace('\n', ' ')
    text = text.replace('_', '')
    text = text.replace('.–', ' ')
    text = text.replace('.—', ' ')   
    text = re.sub(r' +', ' ', text)   
    text = text.strip()
    return text
    

for i, doc in enumerate(docs):
    text = clean_text(doc.page_content)
    
    if not text or len(text) < 20:
        continue

    match = re.search(r'^\[(\d+)\]', text)
    
    section_num = match.group(1) if match else str(i)
    
    parent_data[section_num] = {
        
        "full_text": text,
        "metadata": doc.metadata,
        "children": re.split(r'(?=_Explanation_|_Illustration_)', text)
    }

In [28]:
parent_data['4']['metadata']

{'chapter': 'CHAPTER II',
 'chapter_title': 'OF PUNISHMENTS',
 'section_title': 'Punishments.'}

In [23]:
a = set(parent_data.keys())
len(a) == len(parent_data.keys())

True

In [ ]:
all_children = []

for section_num, data in parent_data.items():
    
    for child in data["children"]:
        all_children.append({
            "child_text": child,
            "parent_id": section_num,
        })

# get the ids
ids = [str(i) for i in range(len(all_children))]

In [247]:
embedding_list = []
for child_data in all_children:
    text = child_data['child_text']
    embedding_list.append(Embedding_model.encode(text))
    

In [248]:
db.add(
    ids=ids,
    embeddings=embedding_list,
    documents=[c["child_text"] for c in all_children],
    metadatas=[{"parent_id": c["parent_id"]} for c in all_children]
)

### small chunkks data starting

In [281]:
# add small chunks to db 
small_chunks_embeddings = []
for chunk in small_chunks:
    small_chunks_embeddings.append(Embedding_model.encode(chunk))

In [ ]:
curr = int(ids[-1])

next_ids  = list(map(str, range(curr+1, curr+1+len(small_chunks))))

In [ ]:

next_ids

In [256]:
user ='what is murder section'
db.query(Embedding_model.encode(user),n_results=5)

{'ids': [['106', '100', '102', '107', '98']],
 'embeddings': None,
 'documents': [['[109] (1) Whoever does any act with such intention or knowledge, and under such circumstances that, if he by that act caused death, he would be guilty of murder, shall be punished with imprisonment of either description for a term which may extend to ten years, and shall also be liable to fine; and if hurt is caused to any person by such act, the offender shall be liable either to imprisonment for life, or to such punishment as is hereinbefore mentioned. (2) When any person offending under sub-section (1) is under sentence of imprisonment for life, he may, if hurt is caused, be punished with death or with imprisonment for life, which shall mean the remainder of that person’s natural life. Illustrations . (a) A shoots at Z with intention to kill him, under such circumstances that, if death ensued, A would be guilty of murder. A is liable to punishment under this section. (b) A, with the intention of caus

# table

In [ ]:
import pdfplumber

file_path = "D:/LegalSaathi AI/data/pdfs/BNS2023.pdf"

all_tables = [] 

with pdfplumber.open(file_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        
        if tables:  # only act if this page actually has tables
            print(f"\n=== Page {page_num} — found {len(tables)} table(s) ===")
            for table_idx, table in enumerate(tables):
                for row in table:
                    print(row)
                all_tables.append({
                    "page": page_num,
                    "table_index": table_idx,
                    "data": table
                })

print(f"\n\nTotal tables found across whole PDF: {len(all_tables)}")



=== Page 19 — found 1 table(s) ===
['Bharatiya Nyaya Sanhita,\n2023', 'Indian Penal Code, 1860']
['CHAPTER I – PRELIMINARY', 'CHAPTER I – INTRODUCTION']
['1. Short title, commencement and\napplication\n1(1)\n1(2)\n1(3)\n1(4)\n1(5)\n1(6)', '1. Title and extent of operation of the\nCode.\nNew Section\n2. Punishment of offences committed\nwithin India.\n3. Punishment of offences committed\nbeyond, but which by law may be\ntried within, India.\n4. Extension of Code to extra-\nterritorial offences.\n5. Certain laws not to be affected by\nthis Act.']
['2. Definitions.', '']

=== Page 21 — found 1 table(s) ===
['3. General explanations\n3(1)\n3(2)\n3(3)\n3(4)\n3(5)', '6. Definitions in the Code to be\nunderstood subject to exceptions.\n7. Sense of expression once\nexplained.\n27. Property in possession of wife,\nclerk or servant.\n32. Words referring to acts include\nillegal omissions.\n34. Acts done by several persons in\nfurtherance of common intention.']

=== Page 22 — found 1 table(s) ==

In [ ]:
import pymupdf

# Step 1: extract just page 56 into a standalone mini-PDF
src = pymupdf.open(r"D:\LegalSaathi AI\data\pdfs\BNS2023.pdf")
single_page_doc = pymupdf.open()  # new empty pdf
single_page_doc.insert_pdf(src, from_page=19, to_page=20)  # 0-indexed, so page 56 = index 55
single_page_doc.save("page56_only.pdf")
single_page_doc.close()
# Step 2: run Docling on just that tiny 1-page PDF

from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("page56_only.pdf")

for table in result.document.tables:
    print(table.export_to_markdown())

[INFO] 2026-06-26 17:38:11,691 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-26 17:38:11,775 [RapidOCR] download_file.py:60: File exists and is valid: D:\LegalSaathi AI\legalAI\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-06-26 17:38:11,778 [RapidOCR] main.py:63: Using D:\LegalSaathi AI\legalAI\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-06-26 17:38:12,054 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-26 17:38:12,064 [RapidOCR] download_file.py:60: File exists and is valid: D:\LegalSaathi AI\legalAI\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-26 17:38:12,066 [RapidOCR] main.py:63: Using D:\LegalSaathi AI\legalAI\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-26 17:38:12,275 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-26 17:38:12,499 [RapidOCR] download_file.py:60: File exists and is val

| Bharatiya Nyaya Sanhita, 2023                     | Indian Penal Code, 1860                                                                  |
|---------------------------------------------------|------------------------------------------------------------------------------------------|
| CHAPTER I - PRELIMINARY                           | CHAPTER I - INTRODUCTION                                                                 |
| 1. Short title, commencement and application 1(1) | 1. Title and extent of operation of the Code.                                            |
| 1(2)                                              | New Section                                                                              |
| 1(3)                                              | 2. Punishment of offences committed within India.                                        |
| 1(4)                                              | 3. Punishment of offences committed beyond, but which by law may be tried wi

In [ ]:
table.export_to_dataframe()

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,"Bharatiya Nyaya Sanhita, 2023","Indian Penal Code, 1860"
0,CHAPTER I - PRELIMINARY,CHAPTER I - INTRODUCTION
1,"1. Short title, commencement and application 1(1)",1. Title and extent of operation of the Code.
2,1(2),New Section
3,1(3),2. Punishment of offences committed within India.
4,1(4),"3. Punishment of offences committed beyond, bu..."
5,1(5),4. Extension of Code to extra- territorial off...
6,1(6),5. Certain laws not to be affected by this Act.
7,2. Definitions.,
8,2(1) 'act',33. 'Act'. 'Omission'
9,2(2) 'animal',47. 'Animal'.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')